# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [2]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：c:\Users\Administrator\Desktop\ecommerce-user-analysis-24012411\data\E Commerce Dataset.xlsx
项目输出目录：c:\Users\Administrator\Desktop\ecommerce-user-analysis-24012411\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [3]:
# 在此写下你的答案：
# 1.一行数据代表一位独立用户，涵盖行为、交易与设备属性
# 2.Churn作为用户流失分析的目标变量
# 3.因为CustomerID是每个用户的身份编号，仅用于区分不同用户。没有计算的意义

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [4]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report=pd.DataFrame({
        "数据类型":data.dtypes,
        "缺失数量":data.isna().sum().sort_values(ascending=False),
        "缺失比例(%)":(data.isna().mean() * 100).round(2),
        "唯一值数量":data.nunique()}
    )
    return report
    # TODO：返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量

# TODO：生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
CashbackAmount,float64,0,0.00,2586
Churn,int64,0,0.00,2
CityTier,int64,0,0.00,3
Complain,int64,0,0.00,2
CouponUsed,float64,256,4.55,17
CustomerID,int64,0,0.00,5630
DaySinceLastOrder,float64,307,5.45,22
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
MaritalStatus,object,0,0.00,3


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [5]:
# TODO：完成项目初始审计
duplicate_rows =raw_df.duplicated().sum()
print("完全重复行数：", duplicate_rows)
duplicate_customer_ids =raw_df["CustomerID"].duplicated().sum()
print("CustomerID 重复数量：", duplicate_customer_ids)
print(raw_df["Churn"].value_counts())
Churn_rate=raw_df["Churn"].mean()
print("流失率：",Churn_rate.round(2))
#
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 0.17

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [6]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [7]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    cleaned_df = raw_df.copy()    #复制数据，避免覆盖原始数据
    logs = []   #创建日志列表 logs
    # ====================== 步骤1：删除完全重复行，并记录日志 ======================
    step_name = "处理完全重复行"
    rule_text = "删除完全重复行，重复记录无增量信息"
    before_cut= cleaned_df.shape[0]
    cleaned_df=cleaned_df.drop_duplicates()
    after_cut=cleaned_df.shape[0]
    affect_cut=before_cut-after_cut

    logs.append({"处理步骤":step_name,
                "处理规则":rule_text,
                "处理前记录数":before_cut,
                "处理后记录数":after_cut,
                "影响记录数":affect_cut
                })
    # ====================== 步骤2：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量 ======================
    step_name = "处理缺失值"
    rule_text = "使用总体中位数补全数值缺失，稳健且不将缺失误判为0，不按Churn分组"
    skip_cols=["CustomerID ","Churn "]
    fill_cols=[col for col in NUMERIC_MISSING_COLS if col not in skip_cols]
    # 填充前总缺失数量
    missing_count = cleaned_df.isna().sum().sort_values(ascending=False)
    before_cnt= missing_count[fill_cols].sum()
    # 逐字段中位数填充
    for col in fill_cols:
        median_val=cleaned_df[col].median()
        cleaned_df[col]=cleaned_df[col].fillna(median_val)
    # 填充后总缺失数量
    after_missing_count=cleaned_df.isna().sum()
    after_cnt=after_missing_count[fill_cols].sum()
    affect_cnt=before_cnt-after_cnt
    logs.append({"处理步骤":step_name,
                "处理规则":rule_text,
                "处理前记录数":int(before_cnt),
                "处理后记录数":int(after_cnt),
                "影响记录数":int(affect_cnt)})
    # ====================== 步骤3：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量 ======================
    step_name = "类别字段标准化映射"
    rule_text = "统一分类字段别名，同类业务名称合并，减少分类冗余"
    pre_row = 0
    post_row = 0
    for col, map_dict in CATEGORY_MAPPINGS.items():
        pre_row += cleaned_df[col].nunique()
    # 遍历映射字典批量替换
    for col, map_dict in CATEGORY_MAPPINGS.items():
        cleaned_df[col] = cleaned_df[col].replace(map_dict)
        post_row += cleaned_df[col].nunique()
    #post_row = cleaned_df.shape[0]
    affect_row = pre_row - post_row

    logs.append({
        "处理步骤": step_name,
        "处理规则": rule_text,
        "处理前记录数": pre_row,
        "处理后记录数": post_row,
        "影响记录数": affect_row
    })

    # ====================== 步骤4：将 Churn 和 Complain 转为整数类型 ======================
    step_name = "字段类型标准化转换"
    rule_text = "将Churn、Complain字段转为整数类型，规范数值存储格式"
    pre_type = cleaned_df.shape[0]
    cleaned_df["Churn"] = cleaned_df["Churn"].astype(int)
    cleaned_df["Complain"] = cleaned_df["Complain"].astype(int)
    post_type = cleaned_df.shape[0]
    affect_type = pre_type - post_type

    logs.append({
        "处理步骤": step_name,
        "处理规则": rule_text,
        "处理前记录数": pre_type,
        "处理后记录数": post_type,
        "影响记录数": affect_type
    })

    # 生成日志表
    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [8]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
#
display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,处理完全重复行,删除完全重复行，重复记录无增量信息,5630,5630,0
1,处理缺失值,使用总体中位数补全数值缺失，稳健且不将缺失误判为0，不按Churn分组,1856,0,1856
2,类别字段标准化映射,统一分类字段别名，同类业务名称合并，减少分类冗余,16,12,4
3,字段类型标准化转换,将Churn、Complain字段转为整数类型，规范数值存储格式,5630,5630,0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [9]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [0, 6, 12, 24, float("inf")]
tenure_labels = ["0-6个月", "6-12个月", "12-24个月", "24个月以上"]
cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=False
)
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
# PreferredLoginDevice映射后移动端统一叫Mobile Phone
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# TODO：生成 outlier_report（每行对应一个待检查字段）
# 需要检测异常的字段
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
report_list = []
for col in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[col])
    row = {"检查字段": col}
    row.update(res)
    report_list.append(row)
outlier_report = pd.DataFrame(report_list)

### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [10]:
# TODO：完成业务规则检查
business_rule_report = pd.DataFrame({
     "规则": ["使用时长小于 0","仓库距离小于 0","订单数小于或等于 0","返现金额小于 0"],
     "不合规记录数": [(cleaned_df["HourSpendOnApp"] < 0).sum(),(cleaned_df["WarehouseToHome"] < 0).sum(), (cleaned_df["OrderCount"] <= 0).sum(),(cleaned_df["CashbackAmount"] < 0).sum()]
})
display(business_rule_report)

# 处理结论：
"""
 1. 业务逻辑：使用时长、配送距离、下单次数、返现金额均不能为负数/零，属于物理上不可能存在的数据，是脏数据；
 2. 若存在不合规行：该类记录无业务意义，会干扰流失分析、分群统计，项目处理方案为直接删除整行；
 3. 若不合规记录数全部为0:说明数据数值符合业务常识，无需删除，可直接用于后续建模分析;
 4. 本报告统一留存至项目清洗日志，完整记录原始数据质量情况，保证流程可追溯。
"""

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


'\n 1. 业务逻辑：使用时长、配送距离、下单次数、返现金额均不能为负数/零，属于物理上不可能存在的数据，是脏数据；\n 2. 若存在不合规行：该类记录无业务意义，会干扰流失分析、分群统计，项目处理方案为直接删除整行；\n 3. 若不合规记录数全部为0:说明数据数值符合业务常识，无需删除，可直接用于后续建模分析;\n 4. 本报告统一留存至项目清洗日志，完整记录原始数据质量情况，保证流程可追溯。\n'

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [11]:
# TODO：完成最终验收
quality_after = build_quality_report(cleaned_df)
#
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)
#
# TODO：导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
#
# TODO：输出 outlier_report 和 business_rule_report
outlier_report.to_csv(f"{OUTPUT_DIR}/outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(f"{OUTPUT_DIR}/business_rule_report.csv", index=False, encoding="utf-8-sig")

# TODO：输出交付文件的路径
print("========== 项目全部交付文件导出完成 ==========")
print(f"输出目录：{OUTPUT_DIR}")
print("1. data_quality_before.csv   清洗前数据质量报告")
print("2. data_quality_after.csv    清洗后数据质量报告")
print("3. cleaning_log.csv          清洗操作日志")
print("4. ecommerce_customer_cleaned.csv 最终清洗完成数据集")
print("5. outlier_report.csv        候选异常值检测报表")
print("6. business_rule_report.csv  业务规则合规检查报表")

========== 项目全部交付文件导出完成 ==========
输出目录：c:\Users\Administrator\Desktop\ecommerce-user-analysis-24012411\output\day04_project
1. data_quality_before.csv   清洗前数据质量报告
2. data_quality_after.csv    清洗后数据质量报告
3. cleaning_log.csv          清洗操作日志
4. ecommerce_customer_cleaned.csv 最终清洗完成数据集
5. outlier_report.csv        候选异常值检测报表
6. business_rule_report.csv  业务规则合规检查报表


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。